# EXP-019: Lựa chọn đặc trưng cho Random Forest

- Ngày: 22/11/2026 (tổng hợp notebook).
- Ngày chạy trong hồ sơ: 22/11/2026.
- Mã thực nghiệm: exp019.
- So với lần trước: exp014 trong mạch trình bày; đối chứng trực tiếp là full68 kiểu exp005 (thay tập đặc trưng).

**Thực nghiệm tương ứng:** [experiment.ipynb](../../experiments/04_additional/EXP-019_RF/experiment.ipynb).

**Mạch thực nghiệm:** [EXP-005](Mai_EXP-005-RF.ipynb) → [EXP-009](Mai_EXP-009_RF_tuning.ipynb) → [EXP-014](Mai_EXP-014_RF_balancing.ipynb) → [EXP-019](Mai_EXP-019_RF_.ipynb).


## 1. Mục tiêu và giả thuyết

**Mục tiêu:** Kiểm tra việc giữ ít đặc trưng quan trọng hơn có duy trì hoặc cải thiện điểm so với dùng đủ 68 biến không.

**Giả thuyết:** Bỏ bớt biến ít hữu ích có thể giảm nhiễu và duy trì hoặc tăng Macro-F1. Hiệu quả có thể khác giữa hai bài toán.


## 2. Cấu hình

Giữ cấu hình tương ứng trong `experiment.ipynb` và hồ sơ lượt chạy.

Hai phiên bản cùng dùng RF 100 cây, `max_depth=None`, `max_features="sqrt"`, seed 42, không cân bằng lớp.

- `full68`: đủ 68 đặc trưng làm đối chứng.
- `selected`: chọn `k ∈ {16, 32, 48, 68}` bằng 3 inner fold theo người. `SelectFromModel` dùng `threshold=-inf`, `max_features=k`; RF xếp hạng biến học từ train.

**Thay đổi:** tập đặc trưng đầu vào. EXP-019 dùng đối chứng RF mặc định full68 kiểu EXP-005, không tiếp tục mô hình cân bằng của EXP-014.


## 3. Cách thực hiện

Dữ liệu physiological68 gồm 6.189 cửa sổ, 68 đặc trưng và 15 người. Nhị phân phân biệt không stress/stress; ba lớp gồm baseline/stress/amusement.

Chỉ dùng outer train để chọn k bằng 3 inner fold. Điền thiếu, xếp hạng và chọn biến đều học trong train của từng inner fold. Sau đó refit trên 14 người và kiểm tra 1 người. Có 30 outer fold, mỗi fold lưu full68 và selected, tổng 60 dòng kết quả.

Mọi xử lý dữ liệu học từ train. Accuracy, Macro-F1 và Balanced Accuracy được tính cho từng người test rồi lấy trung bình đều qua 15 người.


## 4. Kết quả

Accuracy là tỷ lệ dự đoán đúng; Macro-F1 là trung bình F1 các lớp; Balanced Accuracy là trung bình recall các lớp. 

| Bài toán | Phiên bản | Accuracy | F1 (macro) | Balanced Accuracy |
|---|---|---|---|---|
| binary | full68 | 0.8900 | 0.8735 | 0.8865 |
| binary | selected | 0.8916 | 0.8757 | 0.8902 |
| multiclass | full68 | 0.7717 | 0.6480 | 0.6700 |
| multiclass | selected | 0.7323 | 0.6024 | 0.6313 |


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

here = Path.cwd()
repo = next(p for p in (here, *here.parents) if (p / 'src/wesad/evaluation.py').is_file())
source = repo / 'outputs/exp019_rf_mai_20261122'
file = source / ('results.csv' if (source / 'results.csv').exists() else 'metrics_by_subject.csv')
scores = pd.read_csv(file)
if 'model' in scores:
    scores = scores.loc[scores.model.eq('rf')]
if 'variant' not in scores:
    scores['variant'] = 'rf'
keys = ['task', 'variant']
display(scores.groupby(keys)[['macro_f1', 'balanced_accuracy']].agg(['mean', 'std']).round(4))


macro_f1         balanced_accuracy        
                        mean     std              mean     std
task       variant                                            
binary     full68     0.8735  0.1440            0.8865  0.1143
           selected   0.8757  0.1440            0.8902  0.1147
multiclass full68     0.6480  0.1297            0.6700  0.1204
           selected   0.6024  0.1478            0.6313  0.1288

### Đủ 68 biến và chọn đặc trưng

![Đủ 68 biến và chọn đặc trưng](../../experiments/04_additional/EXP-019_RF/figures/rf_full68_vs_selected.png)


### Train và test

![Train và test](../../experiments/04_additional/EXP-019_RF/figures/train_vs_test.png)


### Tần suất lựa chọn đặc trưng

![Tần suất lựa chọn đặc trưng](../../experiments/04_additional/EXP-019_RF/figures/rf_feature_selection_frequency.png)


## 5. So sánh với các lần trước

Các điểm đều là điểm test; chỉ so sánh trong cùng bài toán.

| Bài toán | Lần chạy | Thay đổi | Accuracy | F1 (macro) | Balanced Accuracy |
|---|---|---|---|---|---|
| binary | Đối chứng full68 (kiểu exp005) | Đủ 68 biến | 0.8900 | 0.8735 | 0.8865 |
| binary | exp019, chọn đặc trưng | Chọn k bằng inner CV | 0.8916 | 0.8757 | 0.8902 |
| multiclass | Đối chứng full68 (kiểu exp005) | Đủ 68 biến | 0.7717 | 0.6480 | 0.6700 |
| multiclass | exp019, chọn đặc trưng | Chọn k bằng inner CV | 0.7323 | 0.6024 | 0.6313 |


## 6. Nhận xét

- **Giả thuyết chỉ được ủng hộ nhẹ ở nhị phân:** chọn biến tăng F1 khoảng 0,0022 và BA khoảng 0,0037. Ba lớp giảm F1 khoảng 0,0456 và BA khoảng 0,0387 so với full68.
- **Quan sát bất thường:** trung bình giữ khoảng 47,5 biến ở nhị phân và 53,1 biến ở ba lớp. Đây là số trung bình qua fold, không phải số biến cố định để triển khai. Train vẫn gần 1.
- **Lỗi / khó khăn:** biến tương quan có thể chia sẻ độ quan trọng. Xếp hạng tổng hợp chỉ mô tả kết quả, không dùng để chọn lại biến theo người test. Đối chứng của EXP-019 là full68, không phải mô hình cân bằng EXP-014.


## 7. Bước tiếp theo

1. Tổng hợp bốn hướng RF theo từng bài toán, không mặc định lần sau tốt hơn lần trước.
2. Cân nhắc điểm test, độ biến thiên, số biến và chi phí tính toán.
3. Nếu kết hợp các thay đổi, làm một thực nghiệm riêng; chọn tham số và đặc trưng chỉ bằng dữ liệu train.

**Liên kết:** [EXP-005](Mai_EXP-005-RF.ipynb) → [EXP-009](Mai_EXP-009_RF_tuning.ipynb) → [EXP-014](Mai_EXP-014_RF_balancing.ipynb) → [EXP-019](Mai_EXP-019_RF_.ipynb).
